# Low-Light Image Enhancement

This notebook implements a complete pipeline for low-light image enhancement: (1) analyzing and classifying low vs normal-light images with handcrafted features, (2) enhancing images using classical methods and a lightweight autoencoder, and (3) evaluating all methods with PSNR and SSIM.

## 1. Environment and data check

We use the LOL-v2 (Low-Light) dataset — real-world paired low-light and normal-light images. The dataset is limited to 80–120 pairs for manageable training and evaluation. This section verifies that image pairs are discovered correctly and that images load without errors.

In [ ]:
# Ensure package is importable: run from final-project root after: pip install -e .
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "low").exists():
    PROJECT_ROOT = PROJECT_ROOT / "final-project"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lowlight.config import DATA_DIR, MAX_PAIRS, MIN_PAIRS
from lowlight.data import discover_pairs, load_image

pairs, low_only, normal_only = discover_pairs(str(DATA_DIR))
n_pairs = len(pairs)
print(f"Total pairs: {n_pairs} (required: {MIN_PAIRS}–{MAX_PAIRS})")
print(f"OK: {MIN_PAIRS <= n_pairs <= MAX_PAIRS}")

if pairs:
    path_low, path_normal = pairs[0]
    sh_low = load_image(path_low).shape
    sh_norm = load_image(path_normal).shape
    print(f"Sample low shape: {sh_low}, normal shape: {sh_norm}")

## 2. Preprocessing

- **Resize:** All images are resized to 256×256 for consistent input size and faster training.
- **Normalize:** Pixel values are scaled to [0, 1] for neural networks and consistent metric computation.
- **Augmentation:** Optional horizontal flip or small rotation can be enabled for training; we keep it disabled by default so that evaluation is comparable and reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from lowlight.config import DATA_DIR, IMAGE_SIZE
from lowlight.data import load_dataset
from lowlight.utils import plot_pair

low_imgs, normal_imgs, pair_paths = load_dataset(
    str(DATA_DIR),
    target_size=IMAGE_SIZE,
    normalize=True,
    augment=False,
    seed=42,
)

print(f"Low images:    {low_imgs.shape}   dtype={low_imgs.dtype}   range=[{low_imgs.min():.2f}, {low_imgs.max():.2f}]")
print(f"Normal images: {normal_imgs.shape}   dtype={normal_imgs.dtype}   range=[{normal_imgs.min():.2f}, {normal_imgs.max():.2f}]")

In [ ]:
# Visual check: one pair (low vs normal)
plot_pair(low_imgs[0], normal_imgs[0], title="Preprocessed pair (resized 256×256, normalized [0,1])")

## 3. Phase 1: Image quality analysis and classification

We analyze statistical differences between low-light and normal-light images using handcrafted features and train a binary classifier (Low vs Normal).

- **Features (from grayscale):** Mean intensity, standard deviation (contrast), entropy, and histogram skewness. These capture brightness, contrast, and distribution shape without deep learning.
- **Classifier:** Logistic Regression (simple, interpretable, and sufficient for this feature set).
- **Metrics:** Accuracy, precision, recall, F1-score, and confusion matrix.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from lowlight.features import extract_features_batch

# Extract handcrafted features from grayscale: mean, std, entropy, skewness
X_low = extract_features_batch(low_imgs)      # (N, 4)
X_normal = extract_features_batch(normal_imgs) # (N, 4)
X = np.vstack([X_low, X_normal])
y = np.array([0] * len(low_imgs) + [1] * len(normal_imgs))  # 0 = low-light, 1 = normal-light

# Train/test split (same split can be used for Phase 2/3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training samples: {len(y_train)}, Test samples: {len(y_test)}")
print(f"Feature matrix shape: {X.shape} (samples, features)")

In [ ]:
# Train binary classifier (Logistic Regression)
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Low-light", "Normal-light"]))
print("Metrics summary:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-score:  {f1:.4f}")

In [ ]:
# Confusion matrix visualization
from lowlight.config import FIGURES_DIR
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Low-light", "Normal-light"])
ax.set_yticklabels(["Low-light", "Normal-light"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
plt.colorbar(im, ax=ax, label="Count")
plt.title("Phase 1: Confusion Matrix")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "phase1_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Save trained classifier for reuse
from lowlight.config import MODELS_DIR
import joblib
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(clf, MODELS_DIR / "phase1_classifier.joblib")
print(f"Classifier saved to {MODELS_DIR / 'phase1_classifier.joblib'}")

## 4. Phase 2: Image enhancement

**Part A – Classical methods:** We implement four methods: Histogram Equalization (HE), CLAHE, Gamma Correction, and Single-Scale Retinex (SSR). All operate in the intensity/L channel where appropriate to avoid color shift.

**Part B – Lightweight autoencoder:** A small convolutional autoencoder (several conv layers) is trained to map low-light to normal-light images using MSE loss only, for a modest number of epochs. This provides a learned baseline without GANs or perceptual losses.

In [ ]:
# Pair-level train/test split for Phase 2 & 3 (same random_state for reproducibility)
from sklearn.model_selection import train_test_split
n_pairs = len(low_imgs)
pair_idx = np.arange(n_pairs)
train_idx, test_idx = train_test_split(pair_idx, test_size=0.2, random_state=42)
low_train, low_test = low_imgs[train_idx], low_imgs[test_idx]
normal_train, normal_test = normal_imgs[train_idx], normal_imgs[test_idx]
print(f"Train pairs: {len(train_idx)}, Test pairs: {len(test_idx)}")

In [ ]:
# Phase 2A: Classical enhancement methods
from lowlight.enhancement import enhance_histogram_equalization, enhance_clahe, enhance_gamma, enhance_ssr

def apply_to_batch(imgs, enhance_fn, **kwargs):
    return np.array([enhance_fn(imgs[i], **kwargs) if kwargs else enhance_fn(imgs[i]) for i in range(len(imgs))])

# Apply all four classical methods to test set
enhanced_he   = apply_to_batch(low_test, enhance_histogram_equalization)
enhanced_clahe = apply_to_batch(low_test, enhance_clahe)
enhanced_gamma = apply_to_batch(low_test, enhance_gamma, gamma=0.4)
enhanced_ssr  = apply_to_batch(low_test, enhance_ssr, sigma=35.0)

print("Classical enhancements applied. Shapes:", enhanced_he.shape, enhanced_clahe.shape, enhanced_gamma.shape, enhanced_ssr.shape)

In [ ]:
# Phase 2B: Lightweight autoencoder (MSE loss, 20–30 epochs)
from lowlight.config import AE_EPOCHS, MODELS_DIR
from lowlight.enhancement.autoencoder import train_autoencoder, predict_autoencoder, save_autoencoder

ae_model, ae_losses = train_autoencoder(
    low_train, normal_train,
    epochs=AE_EPOCHS,
    lr=1e-3,
    batch_size=8,
)
print(f"Autoencoder trained for {AE_EPOCHS} epochs. Final loss: {ae_losses[-1]:.6f}")

In [ ]:
# Plot training loss
plt.figure(figsize=(6, 3))
plt.plot(ae_losses, color="tab:blue")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Phase 2B: Autoencoder training loss")
plt.tight_layout()
plt.show()

In [ ]:
# Predict on test set and save model
enhanced_ae = predict_autoencoder(ae_model, low_test)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
save_autoencoder(ae_model, str(MODELS_DIR / "phase2_autoencoder.pt"))
print(f"Enhanced test set shape: {enhanced_ae.shape}. Model saved to {MODELS_DIR / 'phase2_autoencoder.pt'}")

In [ ]:
# Visual comparison: one test image – low | HE | CLAHE | Gamma | SSR | AE | reference
from lowlight.config import FIGURES_DIR
from lowlight.utils import plot_enhancement_comparison
idx = 0
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plot_enhancement_comparison(
    low_test[idx],
    [enhanced_he[idx], enhanced_clahe[idx], enhanced_gamma[idx], enhanced_ssr[idx], enhanced_ae[idx]],
    normal_test[idx],
    ["HE", "CLAHE", "Gamma", "SSR", "Autoencoder"],
    save_path=str(FIGURES_DIR / "phase2_comparison.png"),
)

## 5. Phase 3: Quantitative and qualitative evaluation

We evaluate all methods using PSNR and SSIM against the normal-light reference. Both metrics are standard: PSNR measures pixel-wise fidelity (higher is better), and SSIM captures structural similarity (range [0, 1], higher is better).

**Reported for:** Original low-light input, each classical method (HE, CLAHE, Gamma, SSR), and the autoencoder output—all compared to the same reference.

**Outputs:** A results table (mean ± std), bar charts for PSNR and SSIM, and visual comparisons (input vs enhanced vs reference).

In [ ]:
# Compute PSNR and SSIM for each method
from lowlight.evaluation import psnr, ssim

def compute_metrics_per_image(enhanced_imgs, reference_imgs):
    """enhanced_imgs, reference_imgs: (N, H, W, C). Returns arrays of length N."""
    return (
        np.array([psnr(enhanced_imgs[i], reference_imgs[i]) for i in range(len(enhanced_imgs))]),
        np.array([ssim(enhanced_imgs[i], reference_imgs[i]) for i in range(len(enhanced_imgs))]),
    )

# Reference = normal_test for all
methods = ["Original (low-light)", "Histogram Eq.", "CLAHE", "Gamma", "SSR", "Autoencoder"]
all_enhanced = [low_test, enhanced_he, enhanced_clahe, enhanced_gamma, enhanced_ssr, enhanced_ae]
psnr_means, psnr_stds = [], []
ssim_means, ssim_stds = [], []
for name, imgs in zip(methods, all_enhanced):
    p, s = compute_metrics_per_image(imgs, normal_test)
    psnr_means.append(np.mean(p))
    psnr_stds.append(np.std(p))
    ssim_means.append(np.mean(s))
    ssim_stds.append(np.std(s))
print("Metrics computed for all methods.")

In [ ]:
# Results table
import pandas as pd
results = pd.DataFrame({
    "Method": methods,
    "PSNR (dB)": [f"{m:.2f} ± {sd:.2f}" for m, sd in zip(psnr_means, psnr_stds)],
    "SSIM": [f"{m:.4f} ± {sd:.4f}" for m, sd in zip(ssim_means, ssim_stds)],
})
results

In [ ]:
# Bar charts: PSNR and SSIM across methods
from lowlight.config import FIGURES_DIR
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(methods))
axes[0].bar(x, psnr_means, yerr=psnr_stds, capsize=4, color="steelblue", edgecolor="black")
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, rotation=25, ha="right")
axes[0].set_ylabel("PSNR (dB)")
axes[0].set_title("PSNR vs. reference (higher = better)")
axes[0].set_ylim(bottom=0)
axes[1].bar(x, ssim_means, yerr=ssim_stds, capsize=4, color="coral", edgecolor="black")
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, rotation=25, ha="right")
axes[1].set_ylabel("SSIM")
axes[1].set_title("SSIM vs. reference (higher = better)")
axes[1].set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "phase3_psnr_ssim_bars.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Bar charts saved to {FIGURES_DIR / 'phase3_psnr_ssim_bars.png'}")

### Analysis and discussion (for the report)

The following questions are addressed in the written report:

1. **Why can higher PSNR sometimes correspond to worse visual quality?** PSNR is pixel-wise; oversmoothing or slight misalignment can improve or worsen PSNR without matching human perception.
2. **When do classical methods outperform the autoencoder?** For example when training data is limited, or when the scene is very dark or noisy and the model has not seen similar cases.
3. **Trade-off between brightness and noise:** HE and CLAHE can amplify noise; gamma correction may preserve or alter it differently. The report discusses this trade-off.
4. **Is more brightness always better?** No—overexposure, loss of detail, or unnatural appearance can reduce perceived quality even if metrics improve.

---
## Summary

This notebook implements the full pipeline:

- **Phase 1:** Handcrafted features, binary classifier (Low vs Normal), evaluation metrics, and confusion matrix. Outputs: `figures/phase1_confusion_matrix.png`, `models/phase1_classifier.joblib`.
- **Phase 2:** Four classical methods (HE, CLAHE, Gamma, SSR) and a lightweight autoencoder. Outputs: `figures/phase2_comparison.png`, `models/phase2_autoencoder.pt`.
- **Phase 3:** PSNR and SSIM for all methods, results table, bar charts, and visual comparisons. Output: `figures/phase3_psnr_ssim_bars.png`.